In [5]:
import pandas as pd

# 1. Cargar el dataset (asegúrate de que el nombre coincida con tu archivo)
df = pd.read_csv("data/ventas.csv")

# 2. Ver las primeras 5 filas para entender la estructura
print("--- PRIMERAS FILAS DEL DATASET ---")
display(df.head())

# 3. Ver información técnica (Tipos de datos y nulos)
print("\n--- INFORMACIÓN GENERAL ---")
df.info()

# 4. Ver estadísticas rápidas de las columnas numéricas
print("\n--- ESTADÍSTICAS DESCRIPTIVAS ---")
display(df.describe())

--- PRIMERAS FILAS DEL DATASET ---


,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Age,Gender,Churn
0,44605,2023-05-03 21:30:02,Home,177,1,2427,PayPal,31,1.0,John Rivera,31,Female,0
1,44605,2021-05-16 13:57:44,Electronics,174,3,2448,PayPal,31,1.0,John Rivera,31,Female,0
2,44605,2020-07-13 06:16:57,Books,413,1,2345,Credit Card,31,1.0,John Rivera,31,Female,0
3,44605,2023-01-17 13:14:36,Electronics,396,3,937,Cash,31,0.0,John Rivera,31,Female,0
4,44605,2021-05-01 11:29:27,Books,259,4,2598,PayPal,31,1.0,John Rivera,31,Female,0



--- INFORMACIÓN GENERAL ---
<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Customer ID            250000 non-null  int64  
 1   Purchase Date          250000 non-null  str    
 2   Product Category       250000 non-null  str    
 3   Product Price          250000 non-null  int64  
 4   Quantity               250000 non-null  int64  
 5   Total Purchase Amount  250000 non-null  int64  
 6   Payment Method         250000 non-null  str    
 7   Customer Age           250000 non-null  int64  
 8   Returns                202618 non-null  float64
 9   Customer Name          250000 non-null  str    
 10  Age                    250000 non-null  int64  
 11  Gender                 250000 non-null  str    
 12  Churn                  250000 non-null  int64  
dtypes: float64(1), int64(7), str(5)
memory usage: 24.8 MB

--- ESTADÍSTICAS

,Customer ID,Product Price,Quantity,Total Purchase Amount,Customer Age,Returns,Age,Churn
count,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,202618.000000,250000.000000,250000.00000
mean,25017.632092,254.742724,3.004936,2725.385196,43.798276,0.500824,43.798276,0.20052
std,14412.515718,141.738104,1.414737,1442.576095,15.364915,0.500001,15.364915,0.40039
min,1.000000,10.000000,1.000000,100.000000,18.000000,0.000000,18.000000,0.00000
25%,12590.000000,132.000000,2.000000,1476.000000,30.000000,0.000000,30.000000,0.00000
50%,25011.000000,255.000000,3.000000,2725.000000,44.000000,1.000000,44.000000,0.00000
75%,37441.250000,377.000000,4.000000,3975.000000,57.000000,1.000000,57.000000,0.00000
max,50000.000000,500.000000,5.000000,5350.000000,70.000000,1.000000,70.000000,1.00000


Limpieza y Transformación

In [7]:
# 1. Convertir la columna de fecha a tipo datetime
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])

# 2. Manejar los nulos en la columna Returns
# Asumimos que los valores nulos en devoluciones significan que NO hubo devolución (0)
df['Returns'] = df['Returns'].fillna(0).astype(int)

# 3. Eliminar columnas redundantes o innecesarias
# Eliminamos 'Age' porque ya tenemos 'Customer Age'
if 'Age' in df.columns:
    df = df.drop(columns=['Age'])

# 4. Verificar que los cambios se aplicaron correctamente
print("--- TIPOS DE DATOS CORREGIDOS ---")
df.info()

print("\n--- VALORES NULOS RESTANTES ---")
print(df.isnull().sum())

--- TIPOS DE DATOS CORREGIDOS ---
<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Customer ID            250000 non-null  int64         
 1   Purchase Date          250000 non-null  datetime64[us]
 2   Product Category       250000 non-null  str           
 3   Product Price          250000 non-null  int64         
 4   Quantity               250000 non-null  int64         
 5   Total Purchase Amount  250000 non-null  int64         
 6   Payment Method         250000 non-null  str           
 7   Customer Age           250000 non-null  int64         
 8   Returns                250000 non-null  int64         
 9   Customer Name          250000 non-null  str           
 10  Gender                 250000 non-null  str           
 11  Churn                  250000 non-null  int64         
dtypes: datetime64[us](1),

Segmentación RFM

In [8]:
# 1. Definir el punto de referencia temporal (fecha más reciente del dataset)
fecha_referencia = df['Purchase Date'].max()
print(f"Fecha de referencia para el análisis: {fecha_referencia.date()}")

# 2. Agrupar por cliente para calcular Recencia, Frecuencia y Valor Monetario
rfm = df.groupby('Customer ID').agg({
    # Días transcurridos desde la última compra del cliente hasta la fecha de referencia
    'Purchase Date': lambda x: (fecha_referencia - x.max()).days,
    # Conteo de transacciones únicas por cliente
    'Customer ID': 'count',
    # Suma total de lo gastado por el cliente
    'Total Purchase Amount': 'sum'
}).rename(columns={
    'Purchase Date': 'Recency',
    'Customer ID': 'Frequency',
    'Total Purchase Amount': 'Monetary'
}).reset_index()

# 3. Ver las primeras filas del nuevo dataset RFM
print("\n--- DATASET RFM GENERADO ---")
display(rfm.head())

# 4. Ver estadísticas generales de nuestros clientes bajo RFM
print("\n--- ESTADÍSTICAS DEL COMPORTAMIENTO ---")
display(rfm.describe())

Fecha de referencia para el análisis: 2023-09-13

--- DATASET RFM GENERADO ---


,Customer ID,Recency,Frequency,Monetary
0,1,288,3,6290
1,2,72,6,16481
2,3,222,4,9423
3,4,441,5,7826
4,5,424,5,9769



--- ESTADÍSTICAS DEL COMPORTAMIENTO ---


,Customer ID,Recency,Frequency,Monetary
count,49661.000000,49661.000000,49661.000000,49661.000000
mean,24993.104488,261.335374,5.034131,13719.947222
std,14434.429306,246.804539,2.199399,6811.065854
min,1.000000,0.000000,1.000000,125.000000
25%,12494.000000,77.000000,3.000000,8718.000000
50%,24987.000000,186.000000,5.000000,13008.000000
75%,37492.000000,370.000000,6.000000,17921.000000
max,50000.000000,1351.000000,17.000000,50659.000000


Puntuación y Clasificación de Segmentos

In [9]:
# 1. Crear los puntajes (Scores) del 1 al 3 usando cuantiles (pd.qcut)
# Para Recencia, el valor más bajo (pocos días desde la última compra) es el mejor, por eso invertimos las etiquetas.
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), q=3, labels=[3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=3, labels=[1, 2, 3]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=3, labels=[1, 2, 3]).astype(int)

# 2. Sumar los puntajes para obtener el Score RFM Total
rfm['RFM_Score_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# 3. Definir una función lógica para asignar las etiquetas de comportamiento
def asignar_segmento(puntos):
    if puntos >= 8:
        return 'Campeones / VIP'
    elif puntos >= 5:
        return 'Leales y Activos'
    else:
        return 'Clientes Dormidos / En Riesgo'

# Aplicar la función para crear la columna de segmentos
rfm['Segmento'] = rfm['RFM_Score_Total'].apply(asignar_segmento)

# 4. Ver los resultados y cómo quedó distribuido nuestro mercado
print("--- DISTRIBUCIÓN DE NUESTROS CLIENTES ---")
print(rfm['Segmento'].value_counts())

print("\n--- MUESTRA DEL DATASET FINALIZADO ---")
display(rfm[['Customer ID', 'Recency', 'Frequency', 'Monetary', 'RFM_Score_Total', 'Segmento']].head())

--- DISTRIBUCIÓN DE NUESTROS CLIENTES ---
Segmento
Leales y Activos                 22983
Campeones / VIP                  13546
Clientes Dormidos / En Riesgo    13132
Name: count, dtype: int64

--- MUESTRA DEL DATASET FINALIZADO ---


,Customer ID,Recency,Frequency,Monetary,RFM_Score_Total,Segmento
0,1,288,3,6290,4,Clientes Dormidos / En Riesgo
1,2,72,6,16481,8,Campeones / VIP
2,3,222,4,9423,4,Clientes Dormidos / En Riesgo
3,4,441,5,7826,4,Clientes Dormidos / En Riesgo
4,5,424,5,9769,4,Clientes Dormidos / En Riesgo


Unificar y Exportar Datos

In [10]:
# 1. Obtener la información estática y única de cada cliente
# Para evitar duplicar filas por cliente, agrupamos el dataset original por ID 
# y tomamos sus datos demográficos básicos (que no cambian entre transacciones).
info_clientes = df.groupby('Customer ID').agg({
    'Customer Name': 'first',
    'Customer Age': 'first',
    'Gender': 'first',
    'Churn': 'first'  # Tomamos su estado de Churn actual
}).reset_index()

# 2. Unir (Merge) la información demográfica con los resultados del análisis RFM
clientes_procesados = pd.merge(rfm, info_clientes, on='Customer ID', how='inner')

# 3. Ver una muestra del archivo integrado final
print("--- DATASET DE COMPORTAMIENTO INTEGRADO ---")
display(clientes_procesados.head())

# 4. Exportar a un archivo CSV optimizado para el Dashboard
OUTPUT_FILE = "clientes_procesados.csv"
clientes_procesados.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ ¡Éxito! El archivo '{OUTPUT_FILE}' se ha exportado correctamente en tu carpeta.")
print("Este archivo está optimizado para alimentar nuestra interfaz visual en Streamlit.")

--- DATASET DE COMPORTAMIENTO INTEGRADO ---


,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score_Total,Segmento,Customer Name,Customer Age,Gender,Churn
0,1,288,3,6290,2,1,1,4,Clientes Dormidos / En Riesgo,Dominic Cline,67,Female,0
1,2,72,6,16481,3,2,3,8,Campeones / VIP,Crystal Day,42,Female,0
2,3,222,4,9423,2,1,1,4,Clientes Dormidos / En Riesgo,Joseph Perez,31,Male,0
3,4,441,5,7826,1,2,1,4,Clientes Dormidos / En Riesgo,Wyatt Love,37,Male,0
4,5,424,5,9769,1,2,1,4,Clientes Dormidos / En Riesgo,Shannon Hoffman,24,Female,0



✅ ¡Éxito! El archivo 'clientes_procesados.csv' se ha exportado correctamente en tu carpeta.
Este archivo está optimizado para alimentar nuestra interfaz visual en Streamlit.
